# AVD Setup Analysis

This notebook checks step by step whether your Azure Virtual Desktop environment is correctly configured for **Microsoft Entra Single Sign-On (SSO)**.

All steps run via the **Azure CLI** (`az`). Each cell does exactly one small thing.

**What is checked:**
1. Sign-in / Subscription
2. Find host pools
3. Set variables
4. Host pool configuration
5. RDP properties (`enablerdsaadauth`, `targetisaadjoined`)
6. Session hosts (Entra-joined, status)
7. Windows Cloud Login SP (`isRemoteDesktopProtocolEnabled`)
8. Conditional Access policies
9. Summary / checklist

## 1. Check sign-in
Shows the currently signed-in account and the active subscription.

In [1]:
az account show -o table

EnvironmentName    HomeTenantId                          IsDefault    Name    State    TenantId
-----------------  ------------------------------------  -----------  ------  -------  ------------------------------------
AzureCloud         f71980b2-590a-4de9-90d5-6fbc867da951  True         sub-1   Enabled  f71980b2-590a-4de9-90d5-6fbc867da951


## 2. List subscriptions
All reachable subscriptions in the current tenant.

In [2]:
az account list --query "[].{Name:name, Id:id, Tenant:tenantId}" -o table

Name                            Id                                    Tenant
------------------------------  ------------------------------------  ------------------------------------
sub-mhcore                      09808f31-065f-4231-914d-776c2d6bbe34  f71980b2-590a-4de9-90d5-6fbc867da951
sub-mhodaa                      4aecf0e8-2fe2-4187-bc93-0356bd2676f5  f71980b2-590a-4de9-90d5-6fbc867da951
sub-3                           b1658f1f-33e5-4e48-9401-f66ba5e64cce  f71980b2-590a-4de9-90d5-6fbc867da951
sub-mhuser                      556f9b63-ebc9-4c7e-8437-9a05aa8cdb25  f71980b2-590a-4de9-90d5-6fbc867da951
sub-2                           a0844269-41ae-442c-8277-415f1283d422  f71980b2-590a-4de9-90d5-6fbc867da951
sub-mh4                         98525264-1eb4-493f-983d-16a330caa7f6  f71980b2-590a-4de9-90d5-6fbc867da951
sub-mh3                         9aa72379-2067-4948-b51c-de59f4005d04  f71980b2-590a-4de9-90d5-6fbc867da951
msft-odaa-sub                   8bc36fc3-92f0-44ef-b7f6-0108143de35

## 3. Find host pools
Searches for all AVD host pools in the current tenant via Azure Resource Graph.

In [3]:
az extension add --name resource-graph --only-show-errors
az graph query -q "Resources | where type =~ 'microsoft.desktopvirtualization/hostpools' | project name, resourceGroup, subscriptionId, location" -o table

Count    Total_records
-------  ---------------
1        1


## 4. Set variables
Enter the values from step 3 here. These variables are used in the following cells.

In [4]:
HOSTPOOL="hp-cptdazavdvwan"
RG="rg-cptdazavdvwan"
SUB="ff0bb075-6c44-44ee-bb64-d46ce828c62f"

In [5]:
az account set --subscription "$SUB"
az account show --query "{Subscription:name, Id:id}" -o table

Subscription    Id
--------------  ------------------------------------
sub-1           ff0bb075-6c44-44ee-bb64-d46ce828c62f


## 5. Host pool configuration
Shows type, load balancer and SSO-relevant settings of the host pool.

In [6]:
az desktopvirtualization hostpool show -n "$HOSTPOOL" -g "$RG" --query "{Name:name, Type:hostPoolType, LoadBalancer:loadBalancerType, StartVMOnConnect:startVMOnConnect, MaxSessions:maxSessionLimit}" -o table

Name              Type    LoadBalancer    StartVMOnConnect    MaxSessions
----------------  ------  --------------  ------------------  -------------
hp-cptdazavdvwan  Pooled  BreadthFirst    False               5


## 6. Check RDP properties
Important for SSO: `enablerdsaadauth:i:1` and `targetisaadjoined:i:1` must be set.

In [7]:
az desktopvirtualization hostpool show -n "$HOSTPOOL" -g "$RG" --query "customRdpProperty" -o tsv

drivestoredirect:s:;usbdevicestoredirect:s:;redirectclipboard:i:1;redirectprinters:i:0;audiomode:i:0;videoplaybackmode:i:1;devicestoredirect:s:*;redirectcomports:i:1;redirectsmartcards:i:1;enablecredsspsupport:i:1;redirectwebauthn:i:1;use multimon:i:1;enablerdsaadauth:i:1;keyboardhook:i:2;


## 7. Check session hosts
Status and health of the session hosts in the host pool.

In [ ]:
az rest --method GET \
  --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.DesktopVirtualization/hostPools/$HOSTPOOL/sessionHosts?api-version=2024-04-03" \
  --query "value[].{Name:name, Status:properties.status, Agent:properties.agentVersion, AllowNewSession:properties.allowNewSession}" -o table

'sessionhost' is misspelled or not recognized by the system.

https://aka.ms/cli_ref
Read more about the command in reference docs


: 2

## 8. Windows Cloud Login – isRemoteDesktopProtocolEnabled

> ![Conditional Access policies](images/08-windows-cloud-login/image.png)

Checks on the service principal **Windows Cloud Login** (`270efc09-cd0d-444b-a71f-39af4910ec45`) whether RDP authentication is enabled. Must be `true`.

In [7]:
az rest --method GET --url "https://graph.microsoft.com/beta/servicePrincipals(appId='270efc09-cd0d-444b-a71f-39af4910ec45')/remoteDesktopSecurityConfiguration"

{
  "@odata.context": "https://graph.microsoft.com/beta/$metadata#servicePrincipals('270efc09-cd0d-444b-a71f-39af4910ec45')/remoteDesktopSecurityConfiguration/$entity",
  "approvedClientApps": [],
  "approvedClientApps@odata.context": "https://graph.microsoft.com/beta/$metadata#servicePrincipals('270efc09-cd0d-444b-a71f-39af4910ec45')/remoteDesktopSecurityConfiguration/approvedClientApps",
  "id": "id",
  "isRemoteDesktopProtocolEnabled": true,
  "targetDeviceGroups": [
    {
      "displayName": "AVD-SessionHosts-cptdazavdvwan",
      "id": "92322974-f8b4-4762-a7eb-86578b3eb982"
    }
  ],
  "targetDeviceGroups@odata.context": "https://graph.microsoft.com/beta/$metadata#servicePrincipals('270efc09-cd0d-444b-a71f-39af4910ec45')/remoteDesktopSecurityConfiguration/targetDeviceGroups"
}


### Explanation of the output

The query reads the **`remoteDesktopSecurityConfiguration`** of the service principal *Windows Cloud Login* – the central Entra object for AVD SSO.

| Field | Value | Meaning |
|-------|-------|---------|
| `isRemoteDesktopProtocolEnabled` | `true` | **Task 1:** Tenant is allowed to issue **RDP access tokens** → SSO is possible at all |
| `targetDeviceGroups` | 1 group | **Task 2:** Device groups for which the **consent dialog is hidden** |
| `approvedClientApps` | `[]` | Optional client restriction – empty = default, no restriction |
| `id` | `"id"` | Internal object ID (masked), not relevant |

> ![Conditional Access policies](images/08a-ziel-geraete-gruppe/image2.png)

```mermaid
flowchart TD
    SP["Service Principal<br><b>Windows Cloud Login</b><br>AppId 270efc09-…"]
    SP --> RDSC["remoteDesktopSecurityConfiguration"]

    RDSC --> A["<b>isRemoteDesktopProtocolEnabled = true</b><br>RDP token issuance allowed<br>Task 1 → SSO possible"]
    RDSC --> B["<b>targetDeviceGroups</b><br>hide consent dialog<br>Task 2"]
    RDSC --> C["approvedClientApps = empty<br>no client restriction"]

    B --> G["Dynamic group<br><b>AVD-SessionHosts-cptdazavdvwan</b><br>id 92322974-…"]
    G -->|"Rule: device.displayName -startsWith 'vmavd'"| H1["Session host vmavd-01"]
    G --> H2["Session host vmavd-02"]
    G --> H3["… more vmavd-* hosts"]

    classDef ok fill:#e6f4ea,stroke:#1e7e34,stroke-width:1px,color:#102a13;
    classDef grp fill:#e7f0fb,stroke:#0b5394,stroke-width:1px,color:#0a2540;
    classDef neutral fill:#f3f3f3,stroke:#888,stroke-width:1px,color:#222;
    class A,B ok;
    class G grp;
    class SP,RDSC,C,H1,H2,H3 neutral;
```

This is how sign-in works with these settings:

```mermaid
sequenceDiagram
    participant U as User (client)
    participant E as Microsoft Entra ID
    participant SP as Windows Cloud Login
    participant H as Session host (vmavd-*)

    U->>E: Sign-in / token request
    Note over E,SP: isRemoteDesktopProtocolEnabled = true<br>→ RDP access token is issued
    E-->>U: RDP token (SSO)
    U->>H: Connection with token
    Note over SP,H: Host is in targetDeviceGroups<br>→ NO consent dialog
    H-->>U: Seamless sign-in (SSO)
```

**In short:** Both Entra switches required for SSO are correct – token issuance active (`true`) and the consent dialog is hidden for the `vmavd-*` hosts via the dynamic group.

### 8a. Check target device group (hide consent dialog)
Checks the group referenced in `targetDeviceGroups`: does it still exist and is it a **dynamic** group (Microsoft recommendation)?

In [8]:
az rest --method GET --url "https://graph.microsoft.com/v1.0/groups/92322974-f8b4-4762-a7eb-86578b3eb982" --query "{Name:displayName, MembershipType:membershipRuleProcessingState, Rule:membershipRule, Types:groupTypes}" -o json

{
  "MembershipType": "On",
  "Name": "AVD-SessionHosts-cptdazavdvwan",
  "Rule": "(device.displayName -startsWith \"vmavd\")",
  "Types": [
    "DynamicMembership"
  ]
}


### 8b. Verify session host membership in the target device group

![image.png](images/08-windows-cloud-login/image2.png)

Section 8a confirmed that the group referenced in `targetDeviceGroups` **exists** and is **dynamic**. This step verifies the **actual members** – the session host devices for which the consent dialog *"Allow this Remote Desktop connection?"* is hidden (pre-consent).

> **Note:** this group is **not** an app assignment. The *Users and groups* tab of the *Windows Cloud Login* enterprise app is empty (*"No application assignments found"*) – and that is correct. The target device group is referenced by the **`targetDeviceGroups`** property on the SSO service principal, not by an app-role assignment.

In the portal you reach the group via **Microsoft Entra ID → Devices → Remote connection configuration → Windows Cloud Login → *Target device groups to enable SSO*** (or directly via **Groups → All groups → `AVD-SessionHosts-cptdazavdvwan` → Members**). Each session host appears there as a **Device** object (e.g. `vmavd01`), **not** as a user:

| Column | Example | Meaning |
|--------|---------|---------|
| **Name** | `vmavd01` | Display name of the session host |
| **Type** | `Device` | It is a **device** object (session hosts are devices, not users) |
| **Object Id** | `276f4c1c-…` | ID of the device object in Entra ID |
| **Device Id** | `3a086508-…` | Hardware/device ID – must match `dsregcmd /status` and the `/devices` query (Section 6) |

#### How pre-consent works

1. **`isRemoteDesktopProtocolEnabled = true`** → the tenant may issue **RDP access tokens** (Task 1, Section 8).
2. **`targetDeviceGroups`** → list of *trusted* device groups → the **consent dialog is hidden** (Task 2, Section 8a).
3. A session host must be a **member of one of these groups**. Otherwise the user sees the consent prompt on the first connection (Entra remembers up to **15 hosts for 30 days** before prompting again).
4. **Recommended:** a **dynamic** group with a membership rule (e.g. `device.displayName -startsWith "vmavd"`), so new session hosts are added automatically.

#### Why only devices and no users?

When you open this group under **Groups → Members**, you only see **devices** (the session hosts), never users. There are two reasons:

1. **Functional – it is about the *target*, not the user.** The consent dialog asks *"do you trust **this device** you are connecting to?"* – i.e. the **session host** (`vmavd01`). Pre-consent is therefore a property of the **target device**, not of the person. *Who* may connect (the user) is controlled elsewhere: via the **Desktop Application Group** (role *Desktop Virtualization User*) and Conditional Access.
2. **Technical – dynamic groups are typed.** A dynamic group in Entra ID is either a **Dynamic Device** *or* a **Dynamic User** group, never mixed. Because this group uses a dynamic **device** membership rule (see *Dynamic membership rules*), it can – by definition – only contain device objects. Users are not evaluated by the rule at all.

The three identities involved in AVD SSO are therefore kept separate:

| Object | Managed in | Purpose |
|--------|-----------|---------|
| **Device** (session host `vmavd01`) | *this* group → `targetDeviceGroups` | Pre-consent: hide the consent dialog |
| **User** | Desktop Application Group (role *Desktop Virtualization User*) | Permission to access the AVD resource |
| **Service principal** *Windows Cloud Login* | Entra → Devices → Remote connection configuration | RDP token issuance (`isRemoteDesktopProtocolEnabled`) |

> **Cross-check:** the **Device Id** shown here must match the Device Id from `dsregcmd /status` on the host and the `/devices` query in Section 6 – this confirms it is really the same device.

#### Official documentation

- [Configure SSO – Hide the consent prompt dialog](https://learn.microsoft.com/azure/virtual-desktop/configure-single-sign-on#hide-the-consent-prompt-dialog)
- [Configure the target device groups](https://learn.microsoft.com/azure/virtual-desktop/configure-single-sign-on#configure-the-target-device-groups)
- [Create a dynamic group in Microsoft Entra ID](https://learn.microsoft.com/entra/identity/users/groups-create-rule)
- [Dynamic membership rules for groups](https://learn.microsoft.com/entra/identity/users/groups-dynamic-membership)

#### Live test

The following query lists the **actual members** of the target device group. The session host (e.g. `vmavd01`) must appear here as a device.


In [9]:
# Live test: list the members of the target device group (pre-consent group).
# The session host device (e.g. vmavd01) must be listed here.
az rest --method GET --url "https://graph.microsoft.com/v1.0/groups/92322974-f8b4-4762-a7eb-86578b3eb982/members" --query "value[].{Name:displayName, DeviceId:deviceId, Trust:trustType, OS:operatingSystem, ObjectId:id}" -o table

## 9. Conditional Access policies
Lists CA policies. Look for policies that require a **compliant device** and affect the app *Windows Cloud Login* / *Azure Virtual Desktop*.

> Note: Filtered to policies that target **apps** (`includeApplications` not empty) – like the app view in the portal. Pure user-action policies (e.g. *Security info registration*) are hidden because they are not relevant for AVD SSO.

#### Reauthentication / Sign-in frequency – app exclusion

Source: [Enforce MFA for Azure Virtual Desktop using Conditional Access](https://learn.microsoft.com/azure/virtual-desktop/set-up-mfa).

Behavior per app when reauthentication is enforced:

| App | App ID | Behavior |
|-----|--------|----------|
| **Azure Virtual Desktop** | `9cdead84-…` | Reauth on feed subscribe/refresh and at the **AVD gateway**. After expiry, **background feed refresh + diagnostics fail silently** until the next interactive sign-in. |
| **Windows Cloud Login** | `270efc09-…` | Reauth when **signing in to the session host** (with SSO enabled). |

- The **"Every time"** sign-in frequency option is **only** supported for the **Windows Cloud Login** app (with SSO enabled).
- **Mandatory?** No – there is no hard requirement to exclude the app `9cdead84` from *every* policy.
- **Recommended?** Yes – for **reauthentication / "Every time" policies**, the enforcement should target *Windows Cloud Login* and *Azure Virtual Desktop* should be excluded to avoid silent feed/diagnostics failures. The exclusion in this tenant follows the Microsoft recommendation.
- **Forbidden:** blocking these apps via CA – this breaks SSO or subscribing to the resources (the web client additionally needs access to *Windows 365*).

> ![Conditional Access policies](images/09-conditional-access-policies/image.png)

In [10]:
az rest --method GET --url "https://graph.microsoft.com/v1.0/identity/conditionalAccess/policies" --query "value[?conditions.applications.includeApplications].{Name:displayName, State:state}" -o table

Name                                                                         State
---------------------------------------------------------------------------  -------
Secure password change on high user risk for Microsoft partners and vendors  enabled
Reauthentication on signin risk for Microsoft partners and vendors           enabled
Multifactor authentication for Microsoft partners and vendors                enabled


### 9a. CA policies – detail: grant controls & target apps
Shows the grant controls (e.g. `mfa`, `compliantDevice`) and the included apps per policy. Look for `compliantDevice` in combination with the AVD/SSO apps.


In [11]:
az rest --method GET --url "https://graph.microsoft.com/v1.0/identity/conditionalAccess/policies" --query "value[?conditions.applications.includeApplications].{Name:displayName, State:state, Grant:grantControls.builtInControls, IncludeApps:conditions.applications.includeApplications, ExcludeApps:conditions.applications.excludeApplications}" -o json

[
  {
    "ExcludeApps": [],
    "Grant": [
      "mfa",
      "passwordChange"
    ],
    "IncludeApps": [
      "All"
    ],
    "Name": "Secure password change on high user risk for Microsoft partners and vendors",
    "State": "enabled"
  },
  {
    "ExcludeApps": [
      "9cdead84-a844-4324-93f2-b2e6bb768d07"
    ],
    "Grant": [
      "mfa"
    ],
    "IncludeApps": [
      "All"
    ],
    "Name": "Reauthentication on signin risk for Microsoft partners and vendors",
    "State": "enabled"
  },
  {
    "ExcludeApps": [],
    "Grant": [
      "mfa"
    ],
    "IncludeApps": [
      "All"
    ],
    "Name": "Multifactor authentication for Microsoft partners and vendors",
    "State": "enabled"
  }
]


### 9c. Resolve the excluded app & why the exclusion exists

The output above contains `ExcludeApps` (`excludeApplications`) – an **app GUID** that is *excluded* from the policy. The next cell resolves that GUID to a readable **display name** via its service principal.

#### Why an app is excluded – official background

In AVD/SSO setups the excluded app is almost always one of these:

| Excluded app (App ID) | Why it is excluded | Source |
|-----------------------|--------------------|--------|
| **Azure Virtual Desktop** (`9cdead84-a844-4324-93f2-b2e6bb768d07`) | In **sign-in frequency / "Every time"** policies the *Every time* option is **only supported for *Windows Cloud Login***. If reauthentication is enforced on *Azure Virtual Desktop*, then after the reauth period **background feed refresh and diagnostics upload silently fail** until the next interactive sign-in. Reauth is therefore targeted at *Windows Cloud Login* and *Azure Virtual Desktop* is excluded. | [Configure sign-in frequency](https://learn.microsoft.com/azure/virtual-desktop/set-up-mfa#configure-sign-in-frequency) |
| **Azure Windows VM Sign-In** (`372140e0-b3b7-4226-8ef9-d57986796201`) | For **Entra-joined** session hosts: if you don't want to restrict sign-in to strong methods (e.g. Windows Hello for Business), you exclude this app so the device sign-in itself isn't blocked by the policy. | [Entra-joined session host VMs](https://learn.microsoft.com/azure/virtual-desktop/azure-ad-joined-session-hosts#access-microsoft-entra-joined-vms) |

Key statements from the documentation:

- *"The **Every time** option is only supported when applied to the **Windows Cloud Login** app when single sign-on is enabled."*
- For *Azure Virtual Desktop*: *"Once the reauthentication period is over, **background feed refresh and diagnostics upload silently fails** until a user completes their next interactive sign in to Microsoft Entra."*

So the exclusion is **intentional**: it avoids silent feed/diagnostics failures (AVD app) or unintended sign-in blocks (VM Sign-In app), while keeping the security control on the app where it belongs (*Windows Cloud Login* for the session-host sign-in).

#### Documentation

- [Enforce MFA for AVD – Configure sign-in frequency](https://learn.microsoft.com/azure/virtual-desktop/set-up-mfa#configure-sign-in-frequency)
- [Conditional Access adaptive session lifetime](https://learn.microsoft.com/entra/identity/conditional-access/concept-session-lifetime#user-sign-in-frequency)
- [Microsoft Entra joined session hosts – exclude *Azure Windows VM Sign-In*](https://learn.microsoft.com/azure/virtual-desktop/azure-ad-joined-session-hosts#access-microsoft-entra-joined-vms)

#### Live test – resolve the excluded app names


In [12]:
# Resolve the display name of every app excluded from a CA policy (excludeApplications).
# Special values like "All" or "Office365" are not service principals and are printed as-is.
for APP_ID in $(az rest --method GET --url "https://graph.microsoft.com/v1.0/identity/conditionalAccess/policies" --query "value[].conditions.applications.excludeApplications[]" -o tsv | sort -u); do
  case "$APP_ID" in
    ""|None) continue ;;
  esac
  NAME=$(az rest --method GET --url "https://graph.microsoft.com/v1.0/servicePrincipals(appId='$APP_ID')" --query "displayName" -o tsv 2>/dev/null)
  echo "$APP_ID  ->  ${NAME:-<no service principal / special value>}"
done

9cdead84-a844-4324-93f2-b2e6bb768d07  ->  Azure Virtual Desktop


### 9b. `compliantDevice` & hybrid join – why this can break SSO

The live check above (`grantControls.builtInControls`) is the decisive diagnostic: it shows **which control** a policy enforces against **which app**. The critical one for AVD is `compliantDevice` (and its sibling `domainJoinedDevice`), because it is a property of the **device**, not something the **user** can satisfy interactively.

#### MFA vs. device-state controls

| Control | Satisfied by | Can the user fix it at connect time? |
|---------|--------------|--------------------------------------|
| `mfa` | User completes an MFA prompt | ✅ Yes – interactive |
| `compliantDevice` | Device is `isCompliant = true` in Intune | ❌ No – must already be true |
| `domainJoinedDevice` | Device is Microsoft Entra **hybrid** joined | ❌ No – depends on join state |

When a policy targeting *Windows Cloud Login* (`270efc09-…`) or *Azure Virtual Desktop* (`9cdead84-…`) requires `compliantDevice`, Entra checks the **connecting device's** Entra device record at token time. If that record is missing, not enrolled, or non-compliant, the token request is rejected – there is no prompt that fixes it in the moment.

#### Where hybrid join makes this fragile

With **Microsoft Entra hybrid join** the device identity comes from on-premises AD and is synced to Entra. The compliance signal only flows correctly when several things line up:

- The device is **registered in Entra** and the device record is present (sync from AD DS via Entra Connect).
- The device is **enrolled in Intune** and actively **reporting compliant** (`isCompliant = true`).
- The **client that initiates the AVD connection** is the same device that carries the compliant record – not a jump host, gateway, or unmanaged endpoint.

Hybrid-joined devices frequently fail one of these in practice:

- Device registration/sync is delayed, so Entra has **no device record yet** → evaluated as non-compliant.
- The device is hybrid joined but **not Intune-enrolled** → there is no compliance signal at all.
- Compliance evaluation is **stale** (last check-in too old, policy not yet evaluated) → reported non-compliant.
- The connection runs from an endpoint **outside** the compliant device's context, so the device claim never reaches the policy.

In all of these the grant control `compliantDevice` is **not** met, even though the user and their credentials are perfectly valid.

```mermaid
flowchart TD
    A["User connects to AVD<br>(hybrid-joined client)"] --> B{"CA policy targets<br>AVD/SSO app?"}
    B -->|"Grant = mfa only"| OK["MFA prompt → access granted"]
    B -->|"Grant = compliantDevice"| C{"Entra device record<br>present & isCompliant?"}
    C -->|"Yes"| OK2["Access granted"]
    C -->|"Not registered / not enrolled<br>/ stale / non-compliant"| ERR["AADSTS53000<br>Device is not in required device state: compliant"]

    classDef ok fill:#e6f4ea,stroke:#1e7e34,color:#102a13;
    classDef bad fill:#fce8e6,stroke:#c5221f,color:#5c0b08;
    class OK,OK2 ok;
    class ERR bad;
```

#### Error AADSTS53000

Error **`AADSTS53000: Device is not in required device state: compliant`** is the exact signature of this situation: a Conditional Access policy with the `compliantDevice` grant control matched the AVD sign-in while the (hybrid-joined) device was not reported compliant. It is **not** an RDP, host-pool or MFA problem – it is purely the device-state grant control failing.

- **Screenshot's tenant:** a policy on the AVD/SSO app carried `compliantDevice`, and the connecting hybrid device wasn't compliant → `AADSTS53000`.

---

#### Step 1 – Confirm the error really comes from device compliance

Don't guess: read the actual sign-in event. Every failed sign-in has an **error code** and the list of **applied CA policies** with the grant control that failed.

- **Portal:** Entra ID → **Sign-in logs** → pick the failed sign-in → tab **Conditional Access** → the offending policy shows **Result: Failure** and the control **Require device to be marked as compliant**. Tab **Basic info** shows error code `53000`.
- **CLI** (needs `AuditLog.Read.All` / `Directory.Read.All`). Replace the UPN:

```bash
USER="user@cptazure.org"
az rest --method GET \
  --url "https://graph.microsoft.com/v1.0/auditLogs/signIns?\$top=10&\$filter=userPrincipalName eq '$USER' and conditionalAccessStatus eq 'failure'" \
  --query "value[].{Time:createdDateTime, App:appDisplayName, ErrorCode:status.errorCode, Detail:status.failureReason, FailedPolicies:appliedConditionalAccessPolicies[?result=='failure'].{Policy:displayName, Controls:enforcedGrantControls}}" \
  -o json
```

**Confirmation:** `ErrorCode = 53000` **and** a failed policy whose `Controls` contain `RequireCompliantDevice` → the error is genuinely a non-compliant-device block. If the error code is different (e.g. `50005`, `50074`, `500011`), the cause is **not** device compliance and the steps below won't help.

#### Step 2 – Find and confirm the device state

Once confirmed, inspect the **connecting** device. The key fields are the **join type** (`trustType`) and the **compliance flag** (`isCompliant`).

```bash
DEVICE="vmavd-01"   # name of the client device that fails to connect
az rest --method GET \
  --url "https://graph.microsoft.com/v1.0/devices?\$filter=displayName eq '$DEVICE'" \
  --query "value[].{Name:displayName, Join:trustType, Compliant:isCompliant, Managed:isManaged, OS:operatingSystem, Enabled:accountEnabled, LastSignIn:approximateLastSignInDateTime}" \
  -o json
```

- `trustType`: `AzureAd` = Entra joined, `ServerAd` = Entra **hybrid** joined, `Workplace` = only registered.
- `isCompliant = false` (or `null`) → this is the cause.
- `isManaged = false` → device is **not enrolled in Intune**, so it can never report compliant.

For the Intune view (needs `DeviceManagementManagedDevices.Read.All`) – shows *why* it is non-compliant:

```bash
az rest --method GET \
  --url "https://graph.microsoft.com/v1.0/deviceManagement/managedDevices?\$filter=deviceName eq '$DEVICE'" \
  --query "value[].{Name:deviceName, Compliance:complianceState, Ownership:managedDeviceOwnerType, LastSync:lastSyncDateTime, Enrolled:enrolledDateTime}" \
  -o json
```

> On the device itself, `dsregcmd /status` shows `AzureAdJoined`, `DomainJoined` and `MdmUrl` (enrolled in Intune). This is the fastest local check.

#### Step 3 – Fix the root cause

Pick the action that matches what Step 2 revealed:

| Finding in Step 2 | Fix |
|-------------------|-----|
| Device **not in Entra** (`devices` query empty) | Hybrid join / Entra Connect sync is missing or delayed → verify Entra Connect, run a delta sync, confirm the device object appears. |
| `isManaged = false` (not in Intune) | Enroll the device in Intune (auto-enrollment via GPO for hybrid devices, or manual enrollment), then wait for the first compliance evaluation. |
| `isCompliant = false` (enrolled but failing) | Open the Intune compliance report for the device, remediate the failing setting (BitLocker, AV, min OS version, firewall…), then force **Sync** in the Company Portal / Intune. |
| Compliance is **stale** | Trigger a device check-in (Intune **Sync**, or `dsregcmd /refreshprdc`), then retry after evaluation completes. |
| Device is correct, policy is too broad | Adjust the CA policy: scope it out for AVD, or use an **OR** grant (`Require hybrid joined **OR** compliant`), or **exclude** the apps *Windows Cloud Login* (`270efc09-…`) / *Azure Virtual Desktop* (`9cdead84-…`) from the compliant-device policy. Never *block* these apps. |

#### Step 4 – Verify the fix

Reconnect to AVD and re-run the **Step 1** sign-in-log query. Success looks like `conditionalAccessStatus = success` and no failed policy with `RequireCompliantDevice`. The `isCompliant = true` flag from **Step 2** should now also be set.

## 10. Result – comparison with the Microsoft docs

Comparing my own Entra ID setup for AVD with [Configure single sign-on for Azure Virtual Desktop using Microsoft Entra ID](https://learn.microsoft.com/en-us/azure/virtual-desktop/configure-single-sign-on) (5 tasks):

| # | Task (docs) | Status in tenant | Finding |
|---|-------------|------------------|---------|
| 1 | Enable Entra authentication for RDP | done | Windows Cloud Login (`270efc09-…`) → `isRemoteDesktopProtocolEnabled = true` |
| 2 | Hide the consent dialog | done | `targetDeviceGroups` = `AVD-SessionHosts-cptdazavdvwan`, **dynamic** group, rule `device.displayName -startsWith "vmavd"`, processing `On` |
| 3 | Kerberos server object (only with hybrid join / AD DS) | not applicable | Only relevant for Entra-hybrid-joined hosts or Entra-joined + on-prem DCs. Since my AVD environment is Entra ID only, this is not needed |
| 4 | Review Conditional Access | uncritical | I have 3 app-targeting policies (all `IncludeApps: All`), **none** requires `compliantDevice` (only `mfa` / `passwordChange`). Policy *Reauthentication on signin risk* excludes the app **Azure Virtual Desktop** (`9cdead84-…`). |
| 5 | Host pool RDP property `enablerdsaadauth:i:1` | done | RDP properties setting as expected |
